# Collecting and visualizing data

Based on search_data. But plotting the output.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Add current directory to path so we can import local modules
sys.path.append(os.getcwd())

import search_data
import run_evaluation
from load_input_yaml import load_params
from calculator import cornell_potential_ansatz

# Define data path
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "../data"))

def load_all_runs(root_dir):
    runs = []
    print(f"Searching for runs in {root_dir}...")
    match_paths = search_data.find_matching_runs(root_dir, {})
    print(f"Found {len(match_paths)} runs.")
    
    for path in match_paths:
        try:
            yaml_path = os.path.join(path, "input.yaml")
            metro, gauge = load_params(yaml_path)
            
            # Get results
            calc_res = run_evaluation.get_or_calculate(path)
            
            if "error" in calc_res or "plot_meta" not in calc_res:
                continue
                
            runs.append({
                "path": path,
                "metro": metro,
                "gauge": gauge,
                "meta": calc_res["plot_meta"],
                "results": calc_res
            })
        except Exception as e:
            print(f"Error loading {path}: {e}")
            continue
            
    return runs

# Load data
all_runs = load_all_runs(DATA_DIR)
print(f"Loaded {len(all_runs)} valid runs.")


## V(R) potential and sommer parameter

given a range for beta.
fixed epsilon1 and epsilon2. Fixed L

In [ ]:
# Group data by (L, T, epsilon1, epsilon2) to see variations in beta
groups = defaultdict(list)
for run in all_runs:
    key = (run['metro'].L, run['metro'].T, run['metro'].epsilon1, run['metro'].epsilon2)
    groups[key].append(run)

# Find the group with the most beta values
best_group_key = max(groups, key=lambda k: len(groups[k]))
best_group_runs = groups[best_group_key]
L, T, eps1, eps2 = best_group_key

print(f"Plotting for fixed parameters: L={L}, T={T}, epsilon1={eps1}, epsilon2={eps2}")
print(f"Number of beta values: {len(best_group_runs)}")

# Sort by beta
best_group_runs.sort(key=lambda r: r['metro'].beta)

# Plot V(R) for different beta
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(best_group_runs)))

for i, run in enumerate(best_group_runs):
    beta = run['metro'].beta
    meta = run['meta']
    pots = meta['potentials']
    errs = meta['potential_errors']
    
    rs = sorted([int(k) for k in pots.keys()])
    Vs = [pots[str(r)] for r in rs]
    V_errs = [errs.get(str(r), 0) for r in rs]
    
    # Plot Data Points
    plt.errorbar(rs, Vs, yerr=V_errs, fmt='o', label=f"beta={beta}", color=colors[i], capsize=3)
    
    # Plot Fitted Potential using calculator.py function
    if 'cornell_params' in meta and meta['cornell_params']:
        params = meta['cornell_params']
        # Ensure we have valid parameters
        if params and all(k in params for k in ['A', 'sigma', 'B']):
            r_dense = np.linspace(min(rs), max(rs), 100)
            V_fit = cornell_potential_ansatz(r_dense, params['A'], params['sigma'], params['B'])
            plt.plot(r_dense, V_fit, '--', color=colors[i], alpha=0.6)

plt.xlabel("R/a")
plt.ylabel("aV(R)")
plt.title(f"Static Potential V(R)\nFixed L={L}, T={T}, eps1={eps1}, eps2={eps2}")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot Sommer parameter r0/a vs beta
betas = []
r0_as = []
r0_errs = []

for run in best_group_runs:
    if 'r0' in run['results']:
        betas.append(run['metro'].beta)
        r0_as.append(run['results']['r0'])
        r0_errs.append(run['results'].get('r0_err', 0))

plt.figure(figsize=(8, 5))
plt.errorbar(betas, r0_as, yerr=r0_errs, fmt='o-', capsize=3)
plt.xlabel("beta")
plt.ylabel("r0/a")
plt.title(f"Sommer Parameter r0/a vs beta\nFixed L={L}, T={T}, eps1={eps1}, eps2={eps2}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## V(R) potential and sommer parameter fixed Volume

given a range for beta.
fixed epsilon1 and epsilon2. Fixed Volume -> different L values.

In [ ]:
# Group data by (beta, epsilon1, epsilon2) to see variations in L
l_groups = defaultdict(list)
for run in all_runs:
    key = (run['metro'].beta, run['metro'].epsilon1, run['metro'].epsilon2)
    l_groups[key].append(run)

# Find group with most different L values
best_l_group_key = max(l_groups, key=lambda k: len(set(r['metro'].L for r in l_groups[k])))
best_l_group_runs = l_groups[best_l_group_key]
beta, eps1, eps2 = best_l_group_key

print(f"Plotting for fixed parameters: beta={beta}, epsilon1={eps1}, epsilon2={eps2}")
unique_Ls = sorted(list(set(r['metro'].L for r in best_l_group_runs)))
print(f"Different L values: {unique_Ls}")

# Sort by L
best_l_group_runs.sort(key=lambda r: r['metro'].L)

# Plot V(R) for different L
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(best_l_group_runs)))

for i, run in enumerate(best_l_group_runs):
    L_val = run['metro'].L
    T_val = run['metro'].T 
    meta = run['meta']
    pots = meta['potentials']
    errs = meta['potential_errors']
    
    rs = sorted([int(k) for k in pots.keys()])
    Vs = [pots[str(r)] for r in rs]
    V_errs = [errs.get(str(r), 0) for r in rs]
    
    label = f"L={L_val}, T={T_val}"
    # Plot Data Points
    plt.errorbar(rs, Vs, yerr=V_errs, fmt='o', label=label, color=colors[i], capsize=3)
    
    # Plot Fitted Potential using calculator.py function
    if 'cornell_params' in meta and meta['cornell_params']:
        params = meta['cornell_params']
        if params and all(k in params for k in ['A', 'sigma', 'B']):
            r_dense = np.linspace(min(rs), max(rs), 100)
            V_fit = cornell_potential_ansatz(r_dense, params['A'], params['sigma'], params['B'])
            plt.plot(r_dense, V_fit, '--', color=colors[i], alpha=0.6)

plt.xlabel("R/a")
plt.ylabel("aV(R)")
plt.title(f"Static Potential V(R) for different volumes\nFixed beta={beta}, eps1={eps1}, eps2={eps2}")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot Sommer parameter r0/a vs L (to see finite size effects)
Ls = []
r0_as = []
r0_errs = []

for run in best_l_group_runs:
    if 'r0' in run['results']:
        Ls.append(run['metro'].L)
        r0_as.append(run['results']['r0'])
        r0_errs.append(run['results'].get('r0_err', 0))

plt.figure(figsize=(8, 5))
plt.errorbar(Ls, r0_as, yerr=r0_errs, fmt='o-', capsize=3)
plt.xlabel("L")
plt.ylabel("r0/a")
plt.title(f"Sommer Parameter r0/a vs L\nFixed beta={beta}, eps1={eps1}, eps2={eps2}")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
